**IN:** directory of pdfs we want to extract info from, chromadb vector databases for each of those pdfs

**OUT:** csv that follows the same format as the ERA training data

In [3]:
# TODO expand the code to intake multiple pdfs from a folder instead of just one 

In [4]:
# Import libraries
import chromadb
from openai import OpenAI
from dotenv import load_dotenv
import csv
from datetime import datetime
import re

In [5]:
# Setting the environment

load_dotenv() # get your OPENAI_API_KEY from the .env file

DATA_PATH = r"pdfs"
CHROMA_PATH = r"chroma_db"
PAPER_ID = "BO1005"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(name=PAPER_ID)

pre_prompt = """ 
You are a helpful assistant. You answer questions about diet information in livestock management scientific literature. 
But you only answer based on knowledge I'm providing you. You don't use your internal knowledge and you don't make things up.
If you don't know the answer, just say: I don't know
"""

# TODO edit query to get desired response and so it works for all pdfs
user_query = """
Give a csv table for each diet in this paper. 
Each row will have an ingredient in the diet, the ingredient type, 
the amount of the ingredient, the units for that amount, 
whether the ingredient is dry, and whether it was fed ad libitum.
Ingredient types include Crop Byproduct, Crop Product, Forage Plants, Supplement, and Other Ingredients
"""
# user_query = input("What do you want to know about this paper?\n") # user query can also come from user input

In [6]:
# get chunks from chromadb relevant to user_query
results = collection.query(
    query_texts=[user_query],
    n_results=4 # TODO change amount of results to minimize token usage while keeping good outputs
)
# print(results['documents']) 

# get OpenAI client
client = OpenAI()

# define system prompt with chromadb results 
system_prompt = f"""
{pre_prompt}
--------------------
The data:
{str(results['documents'])}
"""

# send query to llm
response = client.chat.completions.create(
    model="gpt-4o-mini", # TODO check which model is most cost effective
    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_query} 
    ]
)

# print llm response
print("\n\n---------------------\n\n")
print(response.choices[0].message.content)



---------------------


Here are the CSV tables for each diet in the paper:

### Diet TMRL (Leucaena hay and oil seed cake protein)

| Ingredient                   | Ingredient Type   | Amount | Units | Dry | Ad Libitum |
|------------------------------|-------------------|--------|-------|-----|------------|
| Yellow maize meal            | Crop Product      | 22.0   | %     | Yes | Yes        |
| Eragrostis curvula          | Forage Plants     | 18.0   | %     | Yes | Yes        |
| Leucaena leucocephala       | Forage Plants     | 25.0   | %     | Yes | Yes        |
| Wheat bran                   | Crop Byproduct    | 8.0    | %     | Yes | Yes        |
| Cottonseed oil cake meal     | Crop Byproduct    | 9.0    | %     | Yes | Yes        |
| Sunflower oil cake meal      | Crop Byproduct    | 9.0    | %     | Yes | Yes        |
| Full fat soybean meal        | Supplement        | 0.0    | %     | Yes | Yes        |
| Molasses meal                | Other Ingredients  | 7.0    | %  

In [7]:
# TODO output a csv following the ERA training data format that includes data for all input pdfs
output_text = response.choices[0].message.content

log_filename = "llm_log.csv"
with open(log_filename, mode="a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    file.seek(0, 2)
    if file.tell() == 0:
        writer.writerow(["timestamp", "user_query", "llm_response"])
    writer.writerow([datetime.now().isoformat(), user_query, output_text])

print(f"Saved raw response to {log_filename}")

tables = re.findall(r"\|[^\n]+\|\n\|[-|]+\|\n((?:\|[^\n]+\|\n?)+)", output_text)

csv_filename = "diet_tables.csv"

for i, table in enumerate(tables, start=1):
    lines = [re.sub(r"^\||\|$", "", line).strip() for line in table.strip().split("\n") if line.strip()]
    rows = [re.split(r"\s*\|\s*", line) for line in lines]

    header = ["Ingredient", "Ingredient_Type", "Amount", "Units", "Dry", "Fed_Ad_Libitum"]

    with open(csv_filename, mode="a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        file.seek(0, 2)
        if file.tell() == 0:
            writer.writerow(header)
        for row in rows:
            if len(row) == 6:  # only write valid rows
                cleaned = [r.strip().replace(" ", "_") for r in row]
                writer.writerow(cleaned)

print(f"Saved extracted tables to {csv_filename}")

Saved raw response to llm_log.csv
Saved extracted tables to diet_tables.csv
